# LLM Integration with TuiML

This tutorial covers TuiML's LLM integration capabilities:
- **Tool Discovery** - 200+ tools for algorithms, preprocessing, datasets, and workflows
- **Tool Execution** - Programmatic tool calling via `execute_tool()`
- **LLM API Integration** - Format tools for Anthropic, OpenAI, and Gemini

## 1. Available Tools Overview

TuiML exposes 200+ tools organized by category.

In [16]:
from tuiml.llm import get_all_tools, get_tool_count, list_tools_by_category

# Get tool counts by category
counts = get_tool_count()
total = sum(counts.values())
print(f"Total available tools: {total}")

# List tools by category
print("\nTools by category:")
for category, count in counts.items():
    print(f"  {category}: {count} tools")

Total available tools: 204

Tools by category:
  algorithm: 70 tools
  preprocessing: 77 tools
  dataset: 24 tools
  feature: 19 tools
  splitting: 14 tools


In [17]:
# Get all tool definitions
tools = get_all_tools()

# Show first few tools
print("Sample tools:")
for name, tool in list(tools.items())[:10]:
    print(f"  - {name}: {tool.description[:60]}...")

Sample tools:
  - tuiml_algorithm_NaiveBayesClassifier: Naive Bayes classifier using **pluggable probability estimat...
  - tuiml_algorithm_NaiveBayesMultinomialClassifier: Multinomial Naive Bayes classifier for **text** and **discre...
  - tuiml_algorithm_BayesianNetworkClassifier: Bayesian Network classifier using **probabilistic graphical ...
  - tuiml_algorithm_DecisionStumpClassifier: Decision Stump - a one-level decision tree....
  - tuiml_algorithm_C45TreeClassifier: C4.5 Decision Tree classifier....
  - tuiml_algorithm_RandomTreeClassifier: Random Tree classifier - a single randomized decision tree....
  - tuiml_algorithm_RandomForestClassifier: Random Forest classifier - ensemble of random trees....
  - tuiml_algorithm_ReducedErrorPruningTreeClassifier: Reduced Error Pruning Tree classifier....
  - tuiml_algorithm_HoeffdingTreeClassifier: Hoeffding Tree (Very Fast Decision Tree - VFDT) classifier....
  - tuiml_algorithm_LogisticModelTreeClassifier: Logistic Model Trees classif

## 2. Workflow Tools

High-level workflow tools for common ML tasks.

In [18]:
from tuiml.llm import get_workflow_tools, WORKFLOW_TOOLS

# List workflow tools
workflow_tools = get_workflow_tools()

print("Workflow Tools:")
for name, schema in workflow_tools.items():
    print(f"\n{name}")
    print(f"  {schema['description']}")

Workflow Tools:

tuiml_train
  Train a machine learning model. Supports supervised (classifiers, regressors) and unsupervised (clusterers) learning. Returns trained model and metrics.

tuiml_predict
  Make predictions using a trained model on new data.

tuiml_evaluate
  Evaluate a trained model on test data and compute metrics.

tuiml_experiment
  Compare multiple algorithms on a dataset with cross-validation and statistical tests. Supports supervised learning (classification, regression) and unsupervised learning (clustering).

tuiml_upload_data
  Upload dataset content directly (CSV or ARFF text). Returns a file path that can be used with other tools like tuiml_train.

tuiml_save_model
  Copy a trained model to a custom path. Use this when the user wants to save or download a model to a specific location.

tuiml_serve_model
  Start a REST API server to serve a trained model for predictions. Returns the URL with endpoints: POST /predict, POST /models/{id}/predict, GET /health, GET /mo

## 3. Executing Tools

Execute tools programmatically using `execute_tool`.

In [1]:
from tuiml.llm import execute_tool

# Train a model
result = execute_tool(
    "tuiml_train",
    algorithm="RandomForestClassifier",
    data="iris",
    algorithm_params={"n_estimators": 10, "random_state": 42},
    cv=5
)
print(f"Train: {result.get('status')}")
if result.get('metrics'):
    print(f"  Metrics: {result['metrics']}")

# Get component details
details = execute_tool("tuiml_describe", name="RandomForestClassifier")
print(f"\nDescribe: {details.get('name')} ({details.get('type')})")
print(f"  Parameters: {list(details.get('parameters', {}).keys())}")

Train: success
  Metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9395583160800551, 'cv_f1_score_std': 0.04058971815950488}

Describe: RandomForestClassifier (classifier)
  Parameters: ['n_estimators', 'max_features', 'max_depth', 'min_samples_split', 'min_samples_leaf', 'bootstrap', 'oob_score', 'random_state', 'n_jobs']


## 4. Direct Component Tools

Each TuiML component is also exposed as a tool.

In [20]:
from tuiml.llm import get_tool

# Get a specific tool definition
rf_tool = get_tool("tuiml_algorithm_RandomForestClassifier")

print("RandomForestClassifier Tool:")
print(f"  Name: {rf_tool.name}")
print(f"  Description: {rf_tool.description[:100]}...")
print(f"  Input Schema: {list(rf_tool.input_schema.get('properties', {}).keys())}")

RandomForestClassifier Tool:
  Name: tuiml_algorithm_RandomForestClassifier
  Description: Random Forest classifier - ensemble of random trees....
  Input Schema: ['n_estimators', 'max_features', 'max_depth', 'min_samples_split', 'min_samples_leaf', 'bootstrap', 'oob_score', 'random_state', 'n_jobs']


In [21]:
# Execute a component tool directly
result = execute_tool(
    "tuiml_algorithm_C45TreeClassifier",
    confidence_factor=0.25,
    min_instances_per_leaf=2
)

print("C45TreeClassifier Tool Result:")
print(f"  Created: {result.get('status')}")

C45TreeClassifier Tool Result:
  Created: error


## 5. Getting Tools for LLM APIs

Get tool schemas formatted for LLM consumption.

In [22]:
from tuiml.llm import get_tools_for_llm

# Get all tools formatted for LLM tool calling
llm_tools = get_tools_for_llm(format="mcp")

print(f"Tools ready for LLM: {len(llm_tools)}")

# Show sample tool schema
sample_tool = llm_tools[0]
print(f"\nSample tool schema:")
print(f"  Name: {sample_tool['name']}")
print(f"  Description: {sample_tool['description'][:80]}...")
print(f"  Input Schema keys: {list(sample_tool['inputSchema'].keys())}")

Tools ready for LLM: 12

Sample tool schema:
  Name: tuiml_train
  Description: Train a machine learning model. Supports supervised (classifiers, regressors) an...
  Input Schema keys: ['type', 'properties', 'required']


## 6. Discovery Tools

Tools for discovering and learning about TuiML capabilities.

In [23]:
from tuiml.llm import DISCOVERY_TOOLS

print("Discovery Tools:")
for name, schema in DISCOVERY_TOOLS.items():
    print(f"\n{name}:")
    print(f"  {schema['description']}")

Discovery Tools:

tuiml_list:
  List all available TuiML components (algorithms, preprocessors, datasets, features).

tuiml_describe:
  Get detailed information and parameter schema for any TuiML component.

tuiml_search:
  Search for components by keyword in name or description.


## 7. Using TuiML Tools with LLM APIs

TuiML tools work with any LLM provider. Each has a slightly different schema format — use the converters below to adapt.

### 7.1 Tool Format Converters

In [24]:
from tuiml.llm import get_workflow_tools, execute_tool
import json

def convert_to_anthropic_format(tools: dict) -> list:
    """
    Convert TuiML tools to Anthropic Claude format.
    
    Anthropic uses:
    {
        "name": "tool_name",
        "description": "...",
        "input_schema": { JSON Schema }
    }
    """
    anthropic_tools = []
    for name, schema in tools.items():
        anthropic_tools.append({
            "name": name,
            "description": schema["description"],
            "input_schema": schema["inputSchema"]
        })
    return anthropic_tools

def convert_to_openai_format(tools: dict) -> list:
    """
    Convert TuiML tools to OpenAI/ChatGPT format.
    
    OpenAI uses:
    {
        "type": "function",
        "function": {
            "name": "...",
            "description": "...",
            "parameters": { JSON Schema }
        }
    }
    """
    openai_tools = []
    for name, schema in tools.items():
        openai_tools.append({
            "type": "function",
            "function": {
                "name": name,
                "description": schema["description"],
                "parameters": schema["inputSchema"]
            }
        })
    return openai_tools

def convert_to_gemini_format(tools: dict) -> list:
    """
    Convert TuiML tools to Google Gemini format.
    
    Gemini uses function declarations:
    {
        "name": "...",
        "description": "...",
        "parameters": { JSON Schema with OpenAPI 3.0 style }
    }
    """
    gemini_tools = []
    for name, schema in tools.items():
        # Gemini doesn't support $schema key
        params = schema["inputSchema"].copy()
        params.pop("$schema", None)
        
        gemini_tools.append({
            "name": name,
            "description": schema["description"],
            "parameters": params
        })
    return gemini_tools

# Get workflow tools
workflow_tools = get_workflow_tools()

print("Available workflow tools:")
for name in workflow_tools:
    print(f"  - {name}")

Available workflow tools:
  - tuiml_train
  - tuiml_predict
  - tuiml_evaluate
  - tuiml_experiment
  - tuiml_upload_data
  - tuiml_save_model
  - tuiml_serve_model
  - tuiml_stop_server
  - tuiml_server_status
  - tuiml_list
  - tuiml_describe
  - tuiml_search


### 7.2 Anthropic Claude

In [25]:
# Convert tools to Anthropic format
anthropic_tools = convert_to_anthropic_format(workflow_tools)

print("Anthropic tool format example:")
print(json.dumps(anthropic_tools[0], indent=2)[:500] + "...")

Anthropic tool format example:
{
  "name": "tuiml_train",
  "description": "Train a machine learning model. Supports supervised (classifiers, regressors) and unsupervised (clusterers) learning. Returns trained model and metrics.",
  "input_schema": {
    "type": "object",
    "properties": {
      "algorithm": {
        "type": "string",
        "description": "Algorithm class name. Examples:\n- Classifiers: 'RandomForestClassifier', 'SVM', 'NaiveBayesClassifier', 'C45TreeClassifier'\n- Regressors: 'LinearRegressio...


In [26]:
# Full Anthropic Claude example (uncomment to run with your API key)

# import anthropic, os
# client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
# tools = convert_to_anthropic_format(get_workflow_tools())
# messages = [{"role": "user", "content": "Train RandomForest on iris"}]
#
# response = client.messages.create(
#     model="claude-sonnet-4-20250514", max_tokens=4096,
#     tools=tools, messages=messages
# )
#
# # Tool use loop: execute tools and send results back
# while response.stop_reason == "tool_use":
#     tool_uses = [b for b in response.content if b.type == "tool_use"]
#     tool_results = []
#     for tu in tool_uses:
#         result = execute_tool(tu.name, **tu.input)
#         tool_results.append({
#             "type": "tool_result",
#             "tool_use_id": tu.id,
#             "content": json.dumps(result)
#         })
#     messages.append({"role": "assistant", "content": response.content})
#     messages.append({"role": "user", "content": tool_results})
#     response = client.messages.create(
#         model="claude-sonnet-4-20250514", max_tokens=4096,
#         tools=tools, messages=messages
#     )

print("Anthropic example ready (uncomment to run)")

Anthropic example ready (uncomment to run)


### 7.3 OpenAI ChatGPT

In [27]:
# Convert tools to OpenAI format
openai_tools = convert_to_openai_format(workflow_tools)

print("OpenAI tool format example:")
print(json.dumps(openai_tools[0], indent=2)[:500] + "...")

OpenAI tool format example:
{
  "type": "function",
  "function": {
    "name": "tuiml_train",
    "description": "Train a machine learning model. Supports supervised (classifiers, regressors) and unsupervised (clusterers) learning. Returns trained model and metrics.",
    "parameters": {
      "type": "object",
      "properties": {
        "algorithm": {
          "type": "string",
          "description": "Algorithm class name. Examples:\n- Classifiers: 'RandomForestClassifier', 'SVM', 'NaiveBayesClassifier', 'C45Dec...


In [28]:
# Full OpenAI ChatGPT example (uncomment to run with your API key)

# from openai import OpenAI
# import os
# client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
# tools = convert_to_openai_format(get_workflow_tools())
# messages = [{"role": "user", "content": "Compare J48 and NaiveBayes on iris"}]
#
# response = client.chat.completions.create(
#     model="gpt-4o", messages=messages, tools=tools, tool_choice="auto"
# )
# message = response.choices[0].message
#
# # Function calling loop
# while message.tool_calls:
#     messages.append(message)
#     for tc in message.tool_calls:
#         result = execute_tool(tc.function.name, **json.loads(tc.function.arguments))
#         messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
#     response = client.chat.completions.create(
#         model="gpt-4o", messages=messages, tools=tools, tool_choice="auto"
#     )
#     message = response.choices[0].message

print("OpenAI example ready (uncomment to run)")

OpenAI example ready (uncomment to run)


### 7.4 Google Gemini

In [29]:
# Convert tools to Gemini format
gemini_tools = convert_to_gemini_format(workflow_tools)

print("Gemini tool format example:")
print(json.dumps(gemini_tools[0], indent=2)[:500] + "...")

Gemini tool format example:
{
  "name": "tuiml_train",
  "description": "Train a machine learning model. Supports supervised (classifiers, regressors) and unsupervised (clusterers) learning. Returns trained model and metrics.",
  "parameters": {
    "type": "object",
    "properties": {
      "algorithm": {
        "type": "string",
        "description": "Algorithm class name. Examples:\n- Classifiers: 'RandomForestClassifier', 'SVM', 'NaiveBayesClassifier', 'C45TreeClassifier'\n- Regressors: 'LinearRegression'...


In [30]:
# Full Google Gemini example (uncomment to run with your API key)

# from google import genai
# from google.genai import types
# import os
# client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
#
# tuiml_tools = get_workflow_tools()
# declarations = [
#     types.FunctionDeclaration(
#         name=name,
#         description=schema["description"],
#         parameters={k: v for k, v in schema["inputSchema"].items() if k != "$schema"}
#     )
#     for name, schema in tuiml_tools.items()
# ]
# tools = types.Tool(function_declarations=declarations)
#
# response = client.models.generate_content(
#     model="gemini-2.0-flash", contents="Search for tree classifiers",
#     config=types.GenerateContentConfig(tools=[tools])
# )
#
# # Function calling loop
# part = response.candidates[0].content.parts[0]
# while hasattr(part, 'function_call') and part.function_call:
#     result = execute_tool(part.function_call.name, **dict(part.function_call.args or {}))
#     response = client.models.generate_content(
#         model="gemini-2.0-flash",
#         contents=[
#             types.Content(role="user", parts=[types.Part(text="Search for tree classifiers")]),
#             types.Content(role="model", parts=[part]),
#             types.Content(role="user", parts=[
#                 types.Part(function_response=types.FunctionResponse(
#                     name=part.function_call.name, response={"result": result}
#                 ))
#             ]),
#         ],
#         config=types.GenerateContentConfig(tools=[tools])
#     )
#     part = response.candidates[0].content.parts[0]

print("Gemini example ready (uncomment to run)")

Gemini example ready (uncomment to run)


### 7.5 Provider Comparison

| Feature | Anthropic | OpenAI | Gemini |
|---------|-----------|--------|--------|
| Tool Schema Key | `input_schema` | `parameters` | `parameters` |
| Tool Container | Direct list | `{"type": "function", ...}` | `FunctionDeclaration` |
| Result Role | `tool_result` | `tool` role | `function_response` |
| Loop Check | `stop_reason == "tool_use"` | `message.tool_calls` | `function_call` in parts |

## Summary

This tutorial covered:

1. **Tool Discovery** - 200+ tools organized by category
2. **Workflow Tools** - High-level ML workflow tools
3. **Tool Execution** - `execute_tool()` for programmatic use
4. **Direct Component Tools** - Per-algorithm tool access
5. **LLM API Formatting** - `get_tools_for_llm()` for schema export
6. **Discovery Tools** - Search and explore capabilities
7. **Provider Integration** - Anthropic, OpenAI, and Gemini converters

### Key Functions

| Function | Description |
|----------|-------------|
| `get_all_tools()` | Get all 200+ tool definitions |
| `get_workflow_tools()` | Get high-level workflow tools |
| `execute_tool(name, **kwargs)` | Execute any TuiML tool |
| `get_tools_for_llm(format)` | Get tools formatted for LLM APIs |